# 第2章　变化的直觉 —— 斜率就是"速度"

> **本章目标**：用 Python 数值实验直观理解"导数就是瞬时变化率"。你将手算割线斜率、用循环观察 h→0 时斜率的收敛过程，并用 matplotlib 动画可视化割线逼近切线的全过程。读完本章，你将拥有一份可复用的数值微分代码和清晰的导数几何直觉。
> **前置知识**：第 1 章（Python 函数定义、NumPy 数组、matplotlib 绘图模板）

In [ ]:
# 环境要求：Python 3.10+, NumPy, Matplotlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

print("✅ 环境就绪")

---

## 2.1　割线斜率——平均变化率

想象你在高速公路上开车，从第 1 小时到第 3 小时走了 240 公里。你的**平均速度**是 (240−0)/(3−1) = 120 km/h。这个"平均速度"就是割线（secant line）的斜率——它只关心起点和终点，不在乎中间你是加速还是减速。

把路程换成函数 `f(x) = x²`，在点 `x=1` 和 `x=3` 之间画一条直线，这条割线的斜率就是**平均变化率**（average rate of change）。

📐 **定义　平均变化率**：函数 `f(x)` 在区间 `[a, b]` 上的平均变化率 = `(f(b) − f(a)) / (b − a)`。几何上，这是穿过曲线上两点 `(a, f(a))` 和 `(b, f(b))` 的割线斜率。

In [ ]:
def f(x):
    return x ** 2

# 两点手算
x1, x2 = 1, 3
slope = (f(x2) - f(x1)) / (x2 - x1)

print(f"割线斜率 = ({f(x2)} - {f(x1)}) / ({x2} - {x1})")
print(f"           = ({f(x2):.0f} - {f(x1):.0f}) / {x2 - x1}")
print(f"           = {slope:.1f}")

# 画图：曲线 + 割线
x = np.linspace(0, 4, 200)
plt.figure(figsize=(6, 4))
plt.plot(x, f(x), 'b-', linewidth=2, label='f(x) = x²')
plt.plot([x1, x2], [f(x1), f(x2)], 'r--', linewidth=2, label=f'割线 (斜率={slope:.1f})')
plt.plot([x1, x2], [f(x1), f(x2)], 'ro', markersize=6)
plt.xlabel('x'); plt.ylabel('f(x)')
plt.title('割线：连接曲线上两点的直线')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

---

## 2.2　让两点无限逼近——瞬时变化率

要想知道"某一瞬间"的变化速度，直觉的做法是：**让第二个点向第一个点无限靠近**。取固定点 `x=2`，让另一个点在 `x=2+h` 的位置，然后让 `h` 越来越小，观察割线斜率的变化趋势。

📐 **定义　瞬时变化率（导数）**：当 `h → 0` 时，割线斜率的极限值。记作 `f'(x) = lim(h→0) [f(x+h) − f(x)] / h`。几何上它就是**切线**（tangent line）的斜率。

In [ ]:
def secant_slope(f, x, h):
    """计算 f 在 x 处、步长为 h 的割线斜率"""
    return (f(x + h) - f(x)) / h

def f(x):
    return x ** 2

x_target = 2
print(f"f(x)=x² 在 x={x_target} 处的割线斜率收敛：")
print(f"{'h':>10}  {'斜率':>12}  {'与真实值(4)的差距':>20}")
print("-" * 48)

for h in [1.0, 0.5, 0.1, 0.05, 0.01, 0.001, 0.0001]:
    slope = secant_slope(f, x_target, h)
    true_val = 2 * x_target  # f'(x²) = 2x, 在 x=2 处 = 4
    print(f"{h:10.4f}  {slope:12.8f}  {abs(slope - true_val):20.16f}")

这里藏着一个重要的坑：**h 太小反而坏事**。当 `h` 小于约 `1e-12` 时，`f(x+h)` 和 `f(x)` 几乎相等，它们相减时有效数字全部丢失——这就是浮点舍入误差（回顾第 1 章的 `0.1 + 0.2 ≠ 0.3`）。

> ⚠️ **数值微分的黄金法则**：h 太小不行（舍入误差），太大也不行（近似粗糙），一般在 `1e-5` 到 `1e-8` 之间最合适。

🔗 **AI 连接**：PyTorch 的 `autograd` 不需要你手动选 h——它用链式法则（第 13 章）精确计算梯度，完全避开数值微分的精度问题。

In [ ]:
print(f"\n{'h':>12}  {'斜率':>16}")
print("-" * 32)
for h in [1e-10, 1e-11, 1e-12, 1e-13, 1e-14, 1e-15, 1e-16]:
    slope = secant_slope(f, x_target, h)
    print(f"{h:12.1e}  {slope:16.12f}")

print("\n⚠️  h=1e-16 时斜率变成 0.0——完全错误！")

---

## 2.3　动画演示——割线如何变成切线

静态数字不如动态画面直观。用 `matplotlib.animation` 做一个动画：固定一个点，让另一个点沿曲线逐渐靠近它，观察割线如何收敛为切线。

> 💡 **Notebook 提示**：运行动画单元格后，割线（红色虚线）将绕固定点旋转，逐步与绿色切线重合。

In [ ]:
def f(x):
    return x ** 2

x0 = 2                     # 固定点（切点）
x_curve = np.linspace(0, 4, 300)
h_values = np.logspace(0, -2.5, 60)  # 从 h=1 到 h≈0.003，对数均匀分布

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_curve, f(x_curve), 'b-', linewidth=2, label='f(x) = x²')
ax.set_xlim(0.5, 4.5); ax.set_ylim(-2, 18)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.grid(alpha=0.3)

# 切线（h→0 的极限）：f'(x)=2x, 在 x0=2 处切线为 y = 4x - 4
tangent_x = np.array([x0 - 1, x0 + 1])
tangent_y = 4 * tangent_x - 4
ax.plot(tangent_x, tangent_y, 'g-', linewidth=2, label='切线 (y=4x-4)')

secant_line, = ax.plot([], [], 'r--', linewidth=1.5, label='割线')
moving_point, = ax.plot([], [], 'ro', markersize=7)
fixed_point,  = ax.plot([x0], [f(x0)], 'ko', markersize=8, label=f'固定点 x={x0}')
title_text = ax.set_title('')

def update(frame):
    h = h_values[frame]
    x1, y1 = x0, f(x0)
    x2, y2 = x0 + h, f(x0 + h)
    secant_line.set_data([x1, x2], [y1, y2])
    moving_point.set_data([x2], [y2])
    slope = (y2 - y1) / h
    title_text.set_text(f'h = {h:.4f}    割线斜率 = {slope:.4f}    (真实导数 = 4.0)')
    return secant_line, moving_point, title_text

ani = animation.FuncAnimation(fig, update, frames=len(h_values),
                              interval=100, blit=True)
ax.legend(loc='upper left')
HTML(ani.to_jshtml())  # 在 Notebook 中显示动画

---

## 2.4　总结：从割线到切线——导数定义的几何直觉

把前三节的实验串联起来：

1. **割线** = 固定两点求斜率 ≈ 平均变化率（区间上的"平均速度"）
2. **让 h→0** = 第二个点无限逼近第一个点
3. **切线** = h→0 时割线的极限位置 ≈ 瞬时变化率（某一瞬间的"速度"）
4. **导数 f'(x)** = 切线的斜率 = `lim(h→0) [f(x+h) − f(x)] / h`

对于 `f(x) = x²`，实验表明 `f'(2) = 4`。事实上符号计算可以证明 `f'(x) = 2x`（第 10 章将系统学习），在任意点上的瞬时变化率，恰好等于该点 x 坐标的两倍。

📐 **定义　导数（Derivative）**：函数 `f(x)` 在点 `x` 处的导数 `f'(x)`，是当自变量的变化量 `h` 趋近于 0 时，割线斜率的极限值。如果该极限存在，称 `f` 在 `x` 处**可导**（differentiable）。

> ⚠️ **注意**：本章用的是**数值微分**（代入具体 h 近似计算），而 PyTorch 用的是**自动微分**（第 14 章）和**符号微分**的混合体。数值微分直观但不够精确，却是理解微分概念的最佳入口。

🔗 **AI 连接**：深度学习中，"梯度"就是多元函数的导数。SGD 每一步用梯度决定参数更新的方向和大小（第 12 章、第 24 章）。而二阶导数（导数的导数）则描述了"变化本身的变化速度"——这是优化器动量和学习率调度的理论基础。

---

## ✏️ 习题

> 在下方新建代码单元格，完成以下练习。

**1.** （概念）割线斜率和切线斜率的本质区别是什么？用一句话回答。

**2.** （概念）数值微分中，h 是不是越小越好？为什么？

**3.** （代码）对 `f(x) = x³`，在 `x=2` 处重复 2.2 节的实验，打印 `h` 和斜率。已知符号导数 `f'(x) = 3x²`，在 `x=2` 处真实值为 12。你的数值结果收敛到 12 了吗？

**4.** （代码）对 `f(x) = 1/x`，在 `x=1` 处重复实验。已知 `f'(x) = −1/x²`，真实值为 −1。观察斜率收敛到 −1 的过程，并解释为什么斜率是负数。

**5.** （代码）修改 2.3 节的动画：将切点从 `x=2` 改为 `x=3`，切线方程更新为 `y = 6x − 9`（因为 `f'(3) = 6`）。运行动画，确认割线收敛到新切线。

---

> 🔗 **章末钩子**：你已经用数值实验感受到了"导数就是瞬时变化率"。但如果函数有多个输入变量（比如 `f(x, y) = x² + y²`），"变化率"又该对谁求呢？
> 
> 预览：在正式进入多元微积分之前，下一章将先为数据世界打好地基——**标量、向量与张量**，它们是 AI 中所有数据的容器。

> 💡 **提示**：完成本章后，运行 `Kernel → Restart & Run All` 确保所有单元格都能正常执行。